In [163]:
import os
import rasterio as rio
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.path as mpth
from pathlib import Path


import Functions
import importlib

importlib.reload(Functions)

<module 'Functions' from '/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/Functions/Functions.py'>

In [164]:
app_path = Functions.get_input_path() / 'App'
input_path = app_path / 'Documents' / 'csv'

path: /home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA


In [165]:
sigma_TM = pd.DataFrame(pd.read_csv(input_path / 'TimeSeries_sigma.csv'))
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
date_S1 = list(pd.unique(pd.to_datetime(sigma_TM['Date'])))

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band'])
sigma_TM['diff_mean'] = sigma_TM.groupby(['Code', 'Band'])['mean'].diff()
display(type(sigma_TM['Date'].iloc[0]))

pandas.Timestamp

In [166]:
moisture = pd.read_excel('/home/frank/Desktop/f.chiapperino_local'
                        '/VALENCIA/SCUOLA/App/Data/Ground_Campaign/Flevoland_data/Data_25_fields/Average_Soil_moisture.xlsx',
                        header=0)

moisture.set_index('Code',inplace=True)
moisture_date = list(pd.to_datetime(moisture.columns))

df_date_moist = pd.DataFrame({'original_date': moisture_date}).sort_values('original_date')
df_date_S1 = pd.DataFrame({'ref_date': date_S1}).sort_values('ref_date')

new_date_moist = pd.merge_asof(df_date_moist, df_date_S1, 
                                left_on='original_date',
                                right_on='ref_date', direction='backward')
moisture.columns = pd.to_datetime(new_date_moist['ref_date'])
display(type(moisture.columns[0]))

moisture = moisture.reset_index().melt(id_vars='Code', var_name='Date', value_name='Moist')
display(type(moisture['Date'].iloc[0]))


pandas.Timestamp

pandas.Timestamp

In [167]:
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
moisture['Date'] = pd.to_datetime(moisture['Date'])

Try the aplha method: $$SSM_{i+1} \approx \frac{\sigma_{0}^{i+1}}{\sigma_{0}^{i}} SSM_{i}$$

In [168]:
sigma_TM = pd.merge(
    sigma_TM, 
    moisture, 
    on=['Code', 'Date'], 
    how='left'  # Mantiene tutte le righe di DF1 e aggiunge la nuova misura dove coincide
)
display(sigma_TM.iloc[0:50])

,Code,Date_Band,mean,std,Date,Band,Name,Type,diff_mean,Moist
0,1545384,20170104_VH,-15.490180,1.038437,2017-01-04,VH,DB_SB2_C,Potatoes,NaN,NaN
1,1545384,20170110_VH,-14.146261,1.136036,2017-01-10,VH,DB_SB2_C,Potatoes,1.343919,NaN
2,1545384,20170116_VH,-20.622658,0.922879,2017-01-16,VH,DB_SB2_C,Potatoes,-6.476397,NaN
3,1545384,20170122_VH,-22.050701,0.694627,2017-01-22,VH,DB_SB2_C,Potatoes,-1.428043,NaN
4,1545384,20170128_VH,-16.818918,0.731224,2017-01-28,VH,DB_SB2_C,Potatoes,5.231783,NaN
5,1545384,20170203_VH,-16.139719,1.037139,2017-02-03,VH,DB_SB2_C,Potatoes,0.679199,NaN
6,1545384,20170209_VH,-21.579485,0.888368,2017-02-09,VH,DB_SB2_C,Potatoes,-5.439766,NaN
7,1545384,20170215_VH,-15.705711,1.027437,2017-02-15,VH,DB_SB2_C,Potatoes,5.873774,NaN
8,1545384,20170221_VH,-15.243009,1.121444,2017-02-21,VH,DB_SB2_C,Potatoes,0.462702,NaN
9,1545384,20170227_VH,-12.984944,1.140039,2017-02-27,VH,DB_SB2_C,Potatoes,2.258065,NaN


In [169]:
df_rainfall = pd.read_parquet(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/rainfall_total.parquet')
df_rainfall.index = pd.to_datetime(df_rainfall.index, format='%Y-%m-%d')
prima_corrispondenza = next((d for d in df_rainfall.index if np.datetime64(d) == np.datetime64(sigma_TM['Date'].iloc[0])), None)

print(f"Data trovata: {prima_corrispondenza}")
df_filtered = df_rainfall.loc[prima_corrispondenza:]
df_weekly = df_filtered.resample('6D', label='left').sum()


Data trovata: 2017-01-04 00:00:00


In [170]:
from matplotlib.backends.backend_pdf import PdfPages
# 1. Trasformazione del dizionario in DataFrame (Formato Lungo)
# Definisci il nome del file di uscita

sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date']).dt.tz_localize(None)

output_folder = app_path / 'Documents'
output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'TS_diff_mean_Rain.pdf'

with PdfPages(pdf_filename) as pdf:
    unique_rois = sigma_TM['Code'].unique()

    for (roi), group in sigma_TM.groupby(['Code']):
        # 1. Prepariamo la figura e gli assi
        fig, ax1 = plt.subplots(figsize=(10, 6))
        
        # Filtriamo i dati ROI
        roi_data = sigma_TM[sigma_TM['Code'] == roi]
        
        
        # --- SICUREZZA DATE ---
        # Filtriamo le piogge per mostrare solo lo stesso periodo della coerenza
        min_date = roi_data['Date'].min()
        max_date = roi_data['Date'].max()
        rain_filtered = df_weekly[(df_weekly.index >= min_date) & (df_weekly.index <= max_date)]

        # --- ASSE 2: PIOGGIA (Sotto le linee) ---
        ax2 = ax1.twinx()
        # Usiamo zorder=1 per metterlo sullo sfondo
        ax2.bar(rain_filtered.index, rain_filtered.iloc[:,0], 
                color='skyblue', alpha=0.3, label='Rainfall (mm)', width=0.8, zorder=1)
        
        ax2.set_ylabel("Rainfall (mm)", color='tab:blue', fontsize=12, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor='tab:blue')
        
        # Spazio sopra le barre per non coprire le linee (regola il moltiplicatore se serve)
        if not rain_filtered.empty and rain_filtered.iloc[:,0].max() > 0:
            ax2.set_ylim(0, rain_filtered.iloc[:,0].max() * 2.5)
        ax2.invert_yaxis() 


        ax1.set_zorder(ax2.get_zorder() + 1)
        ax1.patch.set_visible(False) # Rende ax1 trasparente per vedere ax2 sotto
        
        for band in roi_data['Band'].unique():
            band_data = roi_data[roi_data['Band'] == band]
            line, = ax1.plot(band_data['Date'], band_data['diff_mean'], 
                             marker='o', markersize=4, linestyle='-', linewidth=2, 
                             label=f'diff_mean {band}', zorder=3)
            
            


        ax1.set_ylabel("diff_mean", fontsize=12, fontweight='bold')
        ax1.set_xlabel("Data", fontsize=12)

        # --- LEGENDA E LAYOUT ---
        # Uniamo le labels di entrambi gli assi
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        
        # Spostiamo la legenda leggermente più a destra e usiamo subplots_adjust
        ax1.legend(h1 + h2, l1 + l2, loc='upper left', bbox_to_anchor=(1.08, 1), borderaxespad=0.)

        plt.title(f"ROI: {roi} - diff_mean vs Rainfall", fontsize=14, pad=20)
        ax1.grid(True, linestyle='--', alpha=0.4)
        
        # Ruotiamo le date sull'asse X
        plt.setp(ax1.get_xticklabels(), rotation=45)
        
        # Invece di tight_layout puro, usiamo margin per far spazio alla legenda
        plt.subplots_adjust(right=0.85, bottom=0.15)
        
        pdf.savefig(fig)
        plt.close(fig)